<a href="https://colab.research.google.com/github/XanderPatterson/4487-IS-Class-Labs/blob/main/assignment_12_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("hotels (2).csv")

X = df.drop(columns=["is_canceled", "reservation_status", "reservation_status_date"])
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

num_cols = X_train.select_dtypes(include=[np.number]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

for col in num_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

for col in cat_cols:
    mode_val = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = pd.DataFrame(
    encoder.fit_transform(X_train[cat_cols]),
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_train.index,)
X_test_cat = pd.DataFrame(
    encoder.transform(X_test[cat_cols]),
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_test.index,)

X_train_proc = pd.concat([X_train[num_cols], X_train_cat], axis=1)
X_test_proc = pd.concat([X_test[num_cols], X_test_cat], axis=1)

print(f"Data shapes after preprocessing -> Train: {X_train_proc.shape}, Test: {X_test_proc.shape}")

Data shapes after preprocessing -> Train: (6582, 139), Test: (2822, 139)


### ✍️ Your Response: 🔧
1. The Hotel dataset has 9,404 rows in total between the 6,582 training rows and 2,822 test rows and it has 139 features after one hot encoding.

2. There are numerical features such as lead time, adr, adults, stays in weekend nights, stays in week nights, previous cancellations, and booking changes. And also there are catigorical features including hotel, market segment, deposit type, customer type, meal, and reserved room type.

3. To clean the dataset I decided to split the data into 70% training and 30% testing sets before moving on so that there wouldn't be data leakage.Then I imputed missing numerical values using the medians and missing categorical values using modes. And this was from the training data for each of them. The I used one hot encoding to convert categorical variables into dummy variables. Then I made sure to drop the columns that would then lead to leakage in the data.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [3]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_proc, y_train)

y_pred_nb = nb_model.predict(X_test_proc)

print("=== Naïve Bayes Classification Report ===")
print(classification_report(y_test, y_pred_nb))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_nb))

=== Naïve Bayes Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.37      0.54      2121
           1       0.34      0.99      0.51       701

    accuracy                           0.53      2822
   macro avg       0.67      0.68      0.53      2822
weighted avg       0.83      0.53      0.53      2822

=== Confusion Matrix ===
[[ 791 1330]
 [   5  696]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The model performed okay with an accuracy around 53%. It was more focused on grabbing all of the data that lead to cancelations. And it did identify 99% of that data correctly. It caught 696 out of the 701. It did however generate a lot of wrong ones thinking that there were over 1300 cancelations when that wasn't the case. Looking at the recall and F1 scores would be the best metrics because it tracks to see how reliably it can track the cancelations so that rooms are just empty.

2. Looking at how this would be useful in business it would be nice to have the real time automated alerts. The Bayes model flagged 99% of actual cancellations, so it can serve as a fast way to find at the moment of the booking a higher risk for cancelation type of reservation. This can also help to send more follow ups to these types of customers. This also could give a chance to do some overbooking for rooms that are leading towards the higher risks of cancelation.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_proc)
X_test_scaled = scaler.transform(X_test_proc)

svm_model = LinearSVC(dual=False, max_iter=2000, random_state=42)
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)

print("=== Support Vector Machine (SVM) Classification Report ===")
print(classification_report(y_test, y_pred_svm))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_svm))

=== Support Vector Machine (SVM) Classification Report ===
              precision    recall  f1-score   support

           0       0.92      0.95      0.94      2121
           1       0.84      0.75      0.79       701

    accuracy                           0.90      2822
   macro avg       0.88      0.85      0.87      2822
weighted avg       0.90      0.90      0.90      2822

=== Confusion Matrix ===
[[2021  100]
 [ 173  528]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. The SVM model performed strongly overall with a better accuracy of around 90%. It had good percision and reall of around 83-84% and 75% for the recall. This is the accuracy that it had for correctly identifying cancellations and being able to correctly identify 528 out of 701 canceled bookings. This helped take out less of the wrong ones being selected and only left 100 false ones compared to the over 1300. The best metric would probably be the Weighted F1 score and the F1 score for Class 1. These would work the best because they give a balnce to the precision of avoiding the false claims and the recall on the true cancelations.

2. In business, this could be used to when it isn't a straight forward answer or the data isn't linear. So comparing different types of reservations, looking at corporate bookings, group reservations, or higher complex deals that are books where the pricing and timing variables could interact with one another. It is also a lot more accurate it getting rid of the false alarms, and so this can make it so that the hotel managers can be more confident in their decisions over the cancellations without overing booking the rooms where peoeple actually wanted to stay.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [5]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42,)

mlp_model.fit(X_train_scaled, y_train)

y_pred_mlp = mlp_model.predict(X_test_scaled)

print("=== Neural Network (MLP) Classification Report ===")
print(classification_report(y_test, y_pred_mlp))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_mlp))

=== Neural Network (MLP) Classification Report ===
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      2121
           1       0.84      0.82      0.83       701

    accuracy                           0.92      2822
   macro avg       0.89      0.89      0.89      2822
weighted avg       0.92      0.92      0.92      2822

=== Confusion Matrix ===
[[2007  114]
 [ 123  578]]


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The neural network model or the mlp model does do better than both the Bayes and the SVM. It had the best accuracy of 92% compared to the around 50% and the 90%. It found 578 out of 701 canceled bookings. It also had a good F1 score that showed that it was better in learning the patterns of the data.

2. Management would be more comfortable using this type of model in the core operational structure for forcasting and even looking at revenue. This gives the best accuracy and it has the goal of room pricing and overbooking optimization. However, it won't give the exact reasons nor tell why it is flagged so it would need something else applied as well to be able to tell exactly why something was flagged.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [6]:
from sklearn.metrics import accuracy_score

acc_nb = accuracy_score(y_test, y_pred_nb)
acc_svm = accuracy_score(y_test, y_pred_svm)
acc_mlp = accuracy_score(y_test, y_pred_mlp)

print("=== Model Accuracy Comparison ===")
print(f"1. Naïve Bayes Accuracy:      {acc_nb:.4f} ({acc_nb*100:.1f}%)")
print(f"2. Support Vector Machine:   {acc_svm:.4f} ({acc_svm*100:.1f}%)")
print(f"3. Neural Network (MLP):     {acc_mlp:.4f} ({acc_mlp*100:.1f}%)")

print("\nSummary: The Neural Network (MLP) model performed best overall with 92% accuracy.")

=== Model Accuracy Comparison ===
1. Naïve Bayes Accuracy:      0.5269 (52.7%)
2. Support Vector Machine:   0.9033 (90.3%)
3. Neural Network (MLP):     0.9160 (91.6%)

Summary: The Neural Network (MLP) model performed best overall with 92% accuracy.


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. Best Accuracy was the Neural Network MLP and it was the best with 91.6% around 92% accuracy, followed closely by SVM at 90.3%, while Naive Bayes was behind at 52.7%. The best training time & ease of use was the  Naïve Bayes. It was the fastest to train and required the least setup since it doesn't need feature scaling. The best interpretability was the Naive Bayes and Linear SVM. They are much easier to explain and interpret than the Neural Network.

2. Yes, I would recommend using the Neural Network MLP model. Even though it is a "black box," its higher accuracy of around 92% and strong recall of around 82% mean it catches the most cancellations while keeping false alarms lower. This allows hotel management to plan rooms and bookings accurately and maximize revenue.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend using the Neural Network MLP model to predict booking cancellations. It outperformed all the others with a 92% accuracy, it finds unpredictable cancellations and uses that to help with room inventory planning and safe overbooking strategies to help with the hotel revenue. Key limitations include model complexity Limitations of it might include the "blackbox" and the change of customer habits overtimes. To help with this, periodic model retraining would be the best option. The performance could be helped by using external data, things like flight cancelations and such.

2. This aligns with my learning outcome by helping show me how to prepare data that I find, build and compare classification models including the ones used in this assignment with Naive Bayes, SVM, and Neural Networks. And using models as such to be able to use proper evaluation metrics, and being able change complex predictions into acutal business strategies to give the best decisions to business or company or even things in my own life.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"